# 01 Baseline Model

First model on `data/features.parquet`: predict `pfas_sum` from the distance/count features to potential PFAS sources.

Every run (params, metrics, model) is tracked in MLflow. Start the MLflow server + database first (from the project root):

```bash
docker compose up -d
```

UI: http://localhost:5050

In [1]:
import mlflow
import numpy as np
import pandas as pd
from sklearn.dummy import DummyRegressor
from sklearn.ensemble import HistGradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, r2_score, root_mean_squared_error
from sklearn.model_selection import GroupShuffleSplit

from pfas_trace.config import DATASET_FEATURES, settings

## MLflow setup
The notebook only talks to the MLflow server over HTTP. The server stores runs in PostgreSQL and model files in a Docker volume (see `docker-compose.yml`).

In [2]:
mlflow.set_tracking_uri(settings.MLFLOW_TRACKING_URI)
mlflow.set_experiment("pfas-baseline")

<Experiment: artifact_location='mlflow-artifacts:/1', creation_time=1792160577316, effective_trace_archival_retention=None, experiment_id='1', last_update_time=1792160577316, lifecycle_stage='active', name='pfas-baseline', tags={}, trace_location=None, workspace='default'>

## Load data
- **Target:** `pfas_sum` is extremely skewed (72% zeros, max ~5e8), so we train on `log1p(pfas_sum)`.
- **Features:** all distance/count columns. `date`, `name`, `city` are identifiers; `lat`/`lon` are left out so the model has to learn from the PFAS sources instead of memorising locations.

In [3]:
df = pd.read_parquet(DATASET_FEATURES)

TARGET = "pfas_sum"
NON_FEATURES = ["date", "lat", "lon", "name", "city", TARGET]

X = df.drop(columns=NON_FEATURES)
y = np.log1p(df[TARGET])
X.shape

(732691, 80)

## Train/test split by station
A station (`name`) is measured many times but always has the same features. A random row split would put the same station in train and test and overestimate performance. `GroupShuffleSplit` keeps all measurements of a station on one side.

In [4]:
splitter = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(splitter.split(X, y, groups=df["name"]))

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
len(X_train), len(X_test)

(588098, 144593)

## Train and log
One helper that fits a model and logs params, metrics and the model itself to MLflow. Metrics are on the log scale.

MLflow saves sklearn models with `skops`, which refuses unknown types. `TreePredictor` (inside gradient boosting) is safe to trust because we train the model ourselves.

MLflow saves sklearn models with `skops`, which refuses unknown types. `TreePredictor` (inside gradient boosting) is safe to trust because we train the model ourselves.

In [5]:
TRUSTED_TYPES = ["sklearn.ensemble._hist_gradient_boosting.predictor.TreePredictor"]

def train_and_log(model, run_name):
    with mlflow.start_run(run_name=run_name):
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)

        metrics = {
            "rmse": root_mean_squared_error(y_test, y_pred),
            "mae": mean_absolute_error(y_test, y_pred),
            "r2": r2_score(y_test, y_pred),
        }
        mlflow.log_params(model.get_params())
        mlflow.log_metrics(metrics)
        mlflow.sklearn.log_model(model, name="model", input_example=X_train.head(5), skops_trusted_types=TRUSTED_TYPES)
    return metrics

**Dummy baseline:** always predicts the training mean. Any real model has to beat this.

In [6]:
train_and_log(DummyRegressor(strategy="mean"), "dummy-mean")

2026/10/17 13:03:34 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run dummy-mean at: http://localhost:5050/#/experiments/1/runs/76a725faca694a8bb71e727ca474245d
🧪 View experiment at: http://localhost:5050/#/experiments/1


{'rmse': 1.7491590836878428,
 'mae': 1.3115718999902963,
 'r2': -3.78544577062101e-06}

**Gradient boosting:** fast on ~700k rows, needs no scaling and handles missing values out of the box.

In [7]:
train_and_log(HistGradientBoostingRegressor(random_state=42), "hist-gradient-boosting")

2026/10/17 13:03:42 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.


🏃 View run hist-gradient-boosting at: http://localhost:5050/#/experiments/1/runs/c85a7a8a5229452eb05bb5483b9a4e97
🧪 View experiment at: http://localhost:5050/#/experiments/1


{'rmse': 1.6221860309133214,
 'mae': 1.1156728614197595,
 'r2': 0.13990912731576588}